# Neu-Optimal Training

Copy-paste this into a Jupyter notebook or Kaggle to train the Neu model.

In [1]:
# Install deps
!pip install -q transformers datasets

In [2]:
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from datasets import load_dataset
from transformers import AutoTokenizer

# Model classes
class RMSNorm(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model))
    def forward(self, x): return F.rms_norm(x, self.weight.shape, self.weight, self.eps)

class DiffClassifier(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model//4), nn.SiLU(),
            nn.Linear(d_model//4, 3))
    def forward(self, x):
        p = F.adaptive_avg_pool1d(x.transpose(1,2), 1).squeeze(-1)
        return self.net(p)

class Cheap(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.SiLU(),
            nn.Linear(d_model*2, d_model))
    def forward(self, x): return self.net(x)

class SSM(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_model*2), nn.SiLU(),
            nn.Linear(d_model*2, d_model))
    def forward(self, x): return self.net(x)

class Attn(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.h, self.d = heads, d_model//heads
        self.qkv = nn.Linear(d_model, 3*d_model)
        self.out = nn.Linear(d_model, d_model)
    def forward(self, x):
        B,T,D = x.shape
        q,k,v = self.qkv(x).reshape(B,T,3,self.h,self.d).unbind(2)
        q,k,v = q.transpose(1,2), k.transpose(1,2), v.transpose(1,2)
        s = 1.0/self.d**0.5
        m = torch.ones(T,T,device=x.device).triu(1).logical_not()
        a = s*torch.einsum('bqhd,bkhd->bhqk',q,k).masked_fill_(~m,-float('inf'))
        return self.out(torch.einsum('bhqk,bkhd->bqhd',F.softmax(a,dim=-1),v).reshape(B,T,D))

class NeuBlock(nn.Module):
    def __init__(self, d_model, heads):
        super().__init__()
        self.norm = RMSNorm(d_model)
        self.cls = DiffClassifier(d_model)
        self.cheap = Cheap(d_model)
        self.ssm = SSM(d_model)
        self.attn = Attn(d_model, heads)
    def forward(self, x, temp=1.0):
        h = self.norm(x)
        g = F.softmax(self.cls(h)/temp, dim=-1)
        y = (g[:,0,:,None]*self.cheap(h) + 
             g[:,1,:,None]*self.ssm(h) + 
             g[:,2,:,None]*self.attn(h))
        return x+y, g

class NeuModel(nn.Module):
    def __init__(self, vocab=50257, d_model=640, layers=10, heads=10):
        super().__init__()
        self.emb = nn.Embedding(vocab, d_model)
        self.blocks = nn.ModuleList([NeuBlock(d_model, heads) for _ in range(layers)])
        self.norm = RMSNorm(d_model)
        self.head = nn.Linear(d_model, vocab, bias=False)
        self.head.weight = self.emb.weight
    def forward(self, x):
        x = self.emb(x)
        gs = []
        for b in self.blocks:
            x, g = b(x)
            gs.append(g)
        return self.head(self.norm(x)), gs

print("Model classes defined!")

Model classes defined!


In [3]:
# Config
CONFIG = dict(d_model=640, layers=10, heads=10)
BATCH, SEQ = 16, 256
STEPS, LR = 1500, 1e-4

print(f"GPUs: {torch.cuda.device_count()}")

GPUs: 1


In [ ]:
# Load data
print("Loading data...")
ds = load_dataset('HuggingFaceFW/fineweb', name='sample-10BT', split='train', streaming=True)
tok = AutoTokenizer.from_pretrained('gpt2')
texts = [item['text'] for i,item in enumerate(ds) if i < 30000]
enc = tok(texts, truncation=True, max_length=SEQ, return_tensors='pt', padding=True)
inp, tgt = enc['input_ids'], enc['input_ids'].roll(-1); tgt[:,-1] = -1
loader = DataLoader(TensorDataset(inp,tgt), batch_size=BATCH, shuffle=True)
print(f"Loaded {len(texts)} samples, {len(loader)} batches")

Loading data...


In [ ]:
# Create model
model = NeuModel(**CONFIG).cuda()
opt = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

Params: 82,417,950


In [ ]:
# Train
model.train()
for step, (x,y) in enumerate(loader):
    if step >= STEPS: break
    x,y = x.cuda(), y.cuda()
    out, gs = model(x)
    loss = loss_fn(out.view(-1,50257), y.view(-1))
    opt.zero_grad(); loss.backward(); torch.nn.utils.clip_grad_norm_(model.parameters(),1); opt.step()
    
    if step % 50 == 0:
        with torch.no_grad():
            g = torch.stack(gs).mean(dim=(0,2)).cpu().numpy()
        print(f"Step {step}/{STEPS} | Loss: {loss.item():.4f} | Gates: c={g[0]:.2f} s={g[1]:.2f} a={g[2]:.2f}")

print("Done!")

: 

In [ ]:
# Save model
torch.save(model.state_dict(), 'neu-optimal.pt')
print("Saved!")

: 